# Automotive Marketing Content LoRA Studio - Full Multimodal Notebook

Demo academica completa para campanas automotrices:

1. Leer un dataset SFT JSON con 200+ ejemplos para fine-tuning de LLM.
2. Fine-tunear un LLM instructivo con Unsloth + LoRA/QLoRA.
3. Comparar LLM base vs LLM fine-tuned.
4. Leer `metadata.csv` + imagenes para DreamBooth LoRA visual.
5. Entrenar un LoRA visual sobre SDXL con Diffusers.
6. Comparar SDXL base vs SDXL con LoRA visual.
7. Generar assets comerciales por placement usando la salida del LLM fine-tuned.

Fuentes esperadas:

```text
data/commercial_campaign_sft/commercial_campaign_sft.json
data/car_campaign_lora/metadata.csv
data/car_campaign_lora/images/
```

Este notebook no incluye tus imagenes ni los 200 ejemplos SFT. Esta preparado para validarlos cuando los copies en `data/`.


## 0. Runtime y GPU

Activa GPU en Colab o Kaggle antes de entrenar. El fine-tuning del LLM y el LoRA visual pueden ejecutarse en sesiones separadas si la GPU no alcanza para ambos flujos en una sola corrida.


In [ ]:
import subprocess

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode == 0:
    print("GPU detectada:")
    print("\n".join(result.stdout.split("\n")[:18]))
else:
    print("No se detecto GPU. Puedes validar datasets, pero activa GPU para entrenar.")


## 1. Instalar dependencias

La instalacion se deja controlada por flags. En Colab/Kaggle puede tomar varios minutos. Si ya instalaste dependencias en la sesion, cambia `INSTALL_DEPENDENCIES = False`.


In [ ]:
INSTALL_DEPENDENCIES = True
INSTALL_UNSLOTH = True
INSTALL_DIFFUSERS_FROM_GITHUB = True

if INSTALL_DEPENDENCIES:
    !pip install -q --upgrade pip
    !pip install -q --upgrade huggingface_hub accelerate transformers datasets trl peft bitsandbytes safetensors pandas matplotlib scikit-learn
    # Colab puede dejar Pillow en un estado inconsistente luego de upgrades parciales.
    # Reinstalarlo sin cache evita errores como: cannot import name '_Ink' from PIL._typing.
    !pip install -q --force-reinstall --no-cache-dir "Pillow>=10.4.0"
    print("Dependencias instaladas. Si Colab ya habia importado PIL/torchvision y el error persiste, reinicia el runtime una vez y continua desde la celda 2.")
    if INSTALL_DIFFUSERS_FROM_GITHUB:
        !pip install -q git+https://github.com/huggingface/diffusers.git
    else:
        !pip install -q --upgrade diffusers
    if INSTALL_UNSLOTH:
        !pip install -q "unsloth[colab] @ git+https://github.com/unslothai/unsloth.git"


## 2. Imports, semilla y rutas

El notebook intenta detectar el root del proyecto en Colab, Kaggle o local. Si tu repo queda en otra ruta, edita `PROJECT_ROOT` manualmente.


In [ ]:
import importlib
import json
import math
import os
import random
import re
import shutil
import time
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display
from PIL import Image

# Sanity check para detectar temprano el problema de Pillow/torchvision observado en Colab.
try:
    import torchvision.transforms  # noqa: F401
except ModuleNotFoundError:
    print("torchvision no esta instalado; se continuara mientras Transformers no lo requiera.")
except ImportError as exc:
    raise ImportError(
        "Pillow/torchvision quedo en un estado inconsistente. "
        "Ejecuta de nuevo la celda de instalacion y reinicia el runtime de Colab si persiste. "
        "Error original: " + str(exc)
    )

SEED = 3407
random.seed(SEED)
torch.manual_seed(SEED)

CANDIDATE_ROOTS = [
    Path.cwd(),
    Path("/content/dmc-tp3-gen-multimodal"),
    Path("/kaggle/working/dmc-tp3-gen-multimodal"),
]
PROJECT_ROOT = next((p for p in CANDIDATE_ROOTS if (p / "data").exists()), Path.cwd())

DATA_ROOT = PROJECT_ROOT / "data"
SFT_DIR = DATA_ROOT / "commercial_campaign_sft"
SFT_JSON_PATH = SFT_DIR / "commercial_campaign_sft.json"
SFT_TRAIN_PATH = SFT_DIR / "train.json"
SFT_EVAL_PATH = SFT_DIR / "eval.json"

VISUAL_DATA_DIR = DATA_ROOT / "car_campaign_lora"
VISUAL_IMAGES_DIR = VISUAL_DATA_DIR / "images"
VISUAL_METADATA_PATH = VISUAL_DATA_DIR / "metadata.csv"

OUTPUT_ROOT = PROJECT_ROOT / "outputs"
LLM_OUTPUT_DIR = OUTPUT_ROOT / "commercial-qwen-lora"
VISUAL_LORA_OUTPUT_DIR = OUTPUT_ROOT / "automotive-lora"
TRAIN_IMAGES_DIR = OUTPUT_ROOT / "automotive_training_images"
GENERATED_ASSETS_DIR = OUTPUT_ROOT / "generated_assets"
EVAL_OUTPUT_DIR = OUTPUT_ROOT / "evaluation"

for path in [SFT_DIR, VISUAL_IMAGES_DIR, LLM_OUTPUT_DIR, VISUAL_LORA_OUTPUT_DIR, TRAIN_IMAGES_DIR, GENERATED_ASSETS_DIR, EVAL_OUTPUT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("LLM SFT JSON:", SFT_JSON_PATH)
print("Visual metadata:", VISUAL_METADATA_PATH)
print("Visual images:", VISUAL_IMAGES_DIR)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

for pkg in ["torch", "transformers", "datasets", "peft", "trl", "diffusers", "accelerate"]:
    try:
        mod = importlib.import_module(pkg)
        print(f"{pkg}: {getattr(mod, '__version__', 'installed')}")
    except Exception as exc:
        print(f"{pkg}: no disponible ({exc})")


## 3. Configuracion del proyecto

Ajusta estos valores cuando definas la marca/modelo real y el trigger word visual. El `TRIGGER_WORD` debe aparecer en captions, prompts y ejemplos del LLM.


In [ ]:
BRAND = "Ford"
MODEL_SERIES = "TBD real model"
TRIGGER_WORD = "REALCARMODEL"
INSTANCE_PROMPT = f"{TRIGGER_WORD} real car model automotive marketing campaign asset"

LLM_BASE_MODEL = "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit"
LLM_FALLBACK_MODEL = "unsloth/Qwen2.5-3B-Instruct"
SDXL_BASE_MODEL = "stabilityai/stable-diffusion-xl-base-1.0"

REQUIRE_MIN_SFT_EXAMPLES = 200
RECOMMENDED_SFT_EXAMPLES = 300
TRAIN_FRACTION = 0.80

RUN_LLM_TRAINING = False      # Cambiar a True cuando el JSON SFT este listo y haya GPU.
RUN_VISUAL_TRAINING = False   # Cambiar a True cuando metadata.csv e imagenes esten listas y haya GPU.
RUN_IMAGE_GENERATION = False  # Cambiar a True luego de tener LoRA visual o para generar con SDXL base.

print({
    "brand": BRAND,
    "model_series": MODEL_SERIES,
    "trigger_word": TRIGGER_WORD,
    "llm_base_model": LLM_BASE_MODEL,
    "sdxl_base_model": SDXL_BASE_MODEL,
    "run_llm_training": RUN_LLM_TRAINING,
    "run_visual_training": RUN_VISUAL_TRAINING,
})


## 4. Contrato del dataset SFT del LLM

El LLM lee un JSON con minimo 200 objetos. Cada objeto debe tener `instruction`, `input` y `output`. El `output` debe ser un JSON con los campos comerciales esperados. `negative_prompt` no es obligatorio: el pipeline visual lo agrega con un valor deterministico por defecto.


In [ ]:
EXPECTED_SFT_OUTPUT_FIELDS = {"strategy", "channel_plan", "ad_copy", "image_prompt", "kpis", "business_note"}

EXAMPLE_SFT_RECORD = {
    "instruction": "Act as an advertising strategist for an automotive dealership. Generate a campaign proposal in JSON.",
    "input": "Goal: Lead Generation | Vehicle: hybrid SUV | Price range: mid-range | Audience: Families 35-44 | Customer sector: urban families | Historical channel: Instagram | City: Miami | Language: English | Duration: 30 Days | Promotion: test drive + financing | ROI: 2.10 | Conversion rate: 0.08 | Engagement: 9",
    "output": {
        "strategy": "Promote safety, family space, and fuel efficiency, closing with a clear invitation to book a test drive.",
        "channel_plan": "Use Instagram for visual awareness and lead generation forms; reinforce with Meta Ads remarketing for interested prospects.",
        "ad_copy": "Give your family more space, technology, and efficiency. Book your test drive today and discover the hybrid SUV built for city life.",
        "image_prompt": "REALCARMODEL real car model in an English Instagram ad for a mid-range hybrid SUV dealership campaign targeting urban families in Miami, bright city background, premium automotive commercial photography, clear space for headline, no readable text",
        "kpis": ["Leads", "Cost per Lead", "Test Drive Bookings", "Conversion Rate", "ROI"],
        "business_note": "Prioritize qualified leads and measure test drive bookings before scaling the media budget."
    }
}

print("Ubicacion esperada:", SFT_JSON_PATH)
print(json.dumps(EXAMPLE_SFT_RECORD, indent=2))


## 5. Cargar y validar JSON SFT

Cuando todavia no exista el JSON, esta celda crea un archivo de ejemplo `_example_record.json` para guiar la carga real, pero no intenta entrenar.


In [ ]:
def load_sft_records(path: Path):
    if not path.exists():
        example_path = path.parent / "_example_record.json"
        example_path.write_text(json.dumps([EXAMPLE_SFT_RECORD], indent=2), encoding="utf-8")
        print(f"No existe {path}")
        print(f"Se creo ejemplo en {example_path}. Reemplazalo por un JSON con {REQUIRE_MIN_SFT_EXAMPLES}+ ejemplos.")
        return []

    data = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(data, list):
        raise ValueError("El dataset SFT debe ser una lista JSON de objetos.")

    validated = []
    errors = []
    for idx, item in enumerate(data):
        if not isinstance(item, dict):
            errors.append((idx, "record is not an object"))
            continue
        for key in ["instruction", "input", "output"]:
            if key not in item:
                errors.append((idx, f"missing {key}"))
        output = item.get("output")
        if isinstance(output, str):
            try:
                output = json.loads(output)
                item = {**item, "output": output}
            except Exception:
                errors.append((idx, "output string is not valid JSON"))
        if not isinstance(item.get("output"), dict):
            errors.append((idx, "output is not an object"))
            continue
        missing = EXPECTED_SFT_OUTPUT_FIELDS - set(item["output"].keys())
        if missing:
            errors.append((idx, f"missing output fields: {sorted(missing)}"))
        validated.append(item)

    if errors:
        print("Errores de validacion encontrados, mostrando maximo 10:")
        for err in errors[:10]:
            print(err)
        raise ValueError(f"Dataset SFT invalido: {len(errors)} errores.")

    print(f"Ejemplos SFT cargados: {len(validated)}")
    if len(validated) < REQUIRE_MIN_SFT_EXAMPLES:
        print(f"Advertencia: se recomiendan minimo {REQUIRE_MIN_SFT_EXAMPLES} ejemplos para el entregable.")
    return validated

sft_records = load_sft_records(SFT_JSON_PATH)
if sft_records:
    display(pd.DataFrame([{
        "instruction": r["instruction"][:80],
        "input": r["input"][:120],
        "output_fields": ", ".join(r["output"].keys()),
    } for r in sft_records[:5]]))


## 6. Split train/eval y formato instruct

El split se guarda en `data/commercial_campaign_sft/train.json` y `eval.json` para que la evaluacion sea reproducible.


In [ ]:
def deterministic_split(records, train_fraction=0.8, seed=3407):
    rows = list(records)
    rng = random.Random(seed)
    rng.shuffle(rows)
    train_size = max(1, int(len(rows) * train_fraction)) if rows else 0
    return rows[:train_size], rows[train_size:]

def format_sft_text(record):
    output_json = json.dumps(record["output"], ensure_ascii=False)
    return (
        "### Instruction:\n"
        f"{record['instruction']}\n\n"
        "### Input:\n"
        f"{record['input']}\n\n"
        "### Response:\n"
        f"{output_json}"
    )

if sft_records:
    train_records, eval_records = deterministic_split(sft_records, TRAIN_FRACTION, SEED)
    SFT_TRAIN_PATH.write_text(json.dumps(train_records, ensure_ascii=False, indent=2), encoding="utf-8")
    SFT_EVAL_PATH.write_text(json.dumps(eval_records, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Train: {len(train_records)} -> {SFT_TRAIN_PATH}")
    print(f"Eval: {len(eval_records)} -> {SFT_EVAL_PATH}")
    print(format_sft_text(train_records[0])[:1200])
else:
    train_records, eval_records = [], []
    print("Sin dataset SFT real todavia. Se omite split.")


## 7. Fine-tuning LLM con Unsloth + LoRA/QLoRA

Activa `RUN_LLM_TRAINING = True` en la seccion de configuracion cuando tengas el JSON con 200+ ejemplos. Esta celda guarda el adapter en `outputs/commercial-qwen-lora/`.


In [ ]:
if RUN_LLM_TRAINING:
    if not sft_records:
        raise RuntimeError("Carga primero commercial_campaign_sft.json con 200+ ejemplos.")
    if not torch.cuda.is_available():
        raise RuntimeError("Se requiere GPU para entrenar el LLM con Unsloth.")

    from datasets import Dataset
    from unsloth import FastLanguageModel
    from trl import SFTConfig, SFTTrainer

    max_seq_length = 2048
    load_in_4bit = True

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=LLM_BASE_MODEL,
        max_seq_length=max_seq_length,
        dtype=None,
        load_in_4bit=load_in_4bit,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
    )

    train_dataset = Dataset.from_list([{"text": format_sft_text(r)} for r in train_records])
    eval_dataset = Dataset.from_list([{"text": format_sft_text(r)} for r in eval_records]) if eval_records else None

    training_args = SFTConfig(
        output_dir=str(LLM_OUTPUT_DIR),
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        num_train_epochs=2,
        warmup_ratio=0.03,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=SEED,
        logging_steps=10,
        save_strategy="epoch",
        eval_strategy="epoch" if eval_dataset is not None else "no",
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        args=training_args,
    )

    train_result = trainer.train()
    trainer.save_model(str(LLM_OUTPUT_DIR))
    tokenizer.save_pretrained(str(LLM_OUTPUT_DIR))
    print("LLM LoRA guardado en:", LLM_OUTPUT_DIR)
    print(train_result)
else:
    print("RUN_LLM_TRAINING=False. Se omite entrenamiento del LLM.")


## 8. Inferencia y evaluacion LLM base vs fine-tuned

Esta seccion compara outputs sobre ejemplos de evaluacion. Si no hay adapter entrenado, solo deja preparado el contrato de evaluacion.


In [ ]:
def extract_json_object(text):
    if isinstance(text, dict):
        return text
    if not isinstance(text, str):
        return None
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except Exception:
        return None

def jaccard(a, b):
    aw = set(re.findall(r"\w+", str(a).lower()))
    bw = set(re.findall(r"\w+", str(b).lower()))
    if not aw and not bw:
        return 1.0
    return len(aw & bw) / max(1, len(aw | bw))

def score_llm_outputs(generated_records):
    rows = []
    for item in generated_records:
        parsed = extract_json_object(item.get("generated", ""))
        expected = item.get("expected", {})
        rows.append({
            "example_id": item.get("example_id"),
            "json_valid": parsed is not None,
            "field_coverage": len(EXPECTED_SFT_OUTPUT_FIELDS & set(parsed.keys())) / len(EXPECTED_SFT_OUTPUT_FIELDS) if parsed else 0.0,
            "jaccard_vs_expected": jaccard(parsed, expected) if parsed else 0.0,
            "latency_sec": item.get("latency_sec"),
        })
    return pd.DataFrame(rows)

print("Evaluador LLM listo.")
print("Adapter esperado:", LLM_OUTPUT_DIR)
print("Existe adapter:", any(LLM_OUTPUT_DIR.glob("*")))


In [ ]:
RUN_LLM_EVALUATION = False
MAX_EVAL_EXAMPLES = 5

def build_generation_prompt(record):
    return (
        "### Instruction:\n"
        f"{record['instruction']}\n\n"
        "### Input:\n"
        f"{record['input']}\n\n"
        "### Response:\n"
    )

if RUN_LLM_EVALUATION:
    if not eval_records:
        raise RuntimeError("No hay eval_records. Carga y separa el dataset SFT primero.")
    if not any(LLM_OUTPUT_DIR.glob("*")):
        raise RuntimeError("No se encontro adapter LLM entrenado. Ejecuta RUN_LLM_TRAINING=True primero.")

    from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

    device_map = "auto"
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32

    base_tokenizer = AutoTokenizer.from_pretrained(LLM_BASE_MODEL)
    base_model = AutoModelForCausalLM.from_pretrained(LLM_BASE_MODEL, torch_dtype=dtype, device_map=device_map)
    base_pipe = pipeline("text-generation", model=base_model, tokenizer=base_tokenizer)

    ft_tokenizer = AutoTokenizer.from_pretrained(LLM_OUTPUT_DIR)
    ft_model = AutoModelForCausalLM.from_pretrained(LLM_OUTPUT_DIR, torch_dtype=dtype, device_map=device_map)
    ft_pipe = pipeline("text-generation", model=ft_model, tokenizer=ft_tokenizer)

    comparison = []
    for idx, record in enumerate(eval_records[:MAX_EVAL_EXAMPLES]):
        prompt = build_generation_prompt(record)
        for model_label, gen_pipe in [("base", base_pipe), ("fine_tuned", ft_pipe)]:
            start = time.perf_counter()
            out = gen_pipe(prompt, max_new_tokens=450, do_sample=False, return_full_text=False)[0]["generated_text"]
            latency = time.perf_counter() - start
            comparison.append({
                "example_id": idx,
                "model": model_label,
                "generated": out,
                "expected": record["output"],
                "latency_sec": latency,
            })

    llm_eval_path = EVAL_OUTPUT_DIR / "llm_base_vs_finetuned_outputs.json"
    llm_eval_path.write_text(json.dumps(comparison, ensure_ascii=False, indent=2), encoding="utf-8")
    metrics_df = score_llm_outputs(comparison)
    metrics_path = EVAL_OUTPUT_DIR / "llm_evaluation_report.csv"
    metrics_df.to_csv(metrics_path, index=False)
    display(metrics_df)
    print("Outputs:", llm_eval_path)
    print("Metricas:", metrics_path)
else:
    print("RUN_LLM_EVALUATION=False. Se omite comparacion LLM.")


## 9. Contrato del dataset visual para Diffusion

El modelo de difusion lee un CSV y una carpeta de imagenes. El CSV debe tener `file_path` y `caption`; `file_path` puede ser relativo a `data/car_campaign_lora/`.


In [ ]:
SUPPORTED_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp"}

EXAMPLE_METADATA_CSV = """file_path,caption
./images/real_car_model_01.png,"REALCARMODEL real car model, front three quarter view, metallic blue paint, studio automotive photography, premium lighting"
./images/real_car_model_02.png,"REALCARMODEL real car model, side profile, urban night background, cinematic reflections, commercial car advertisement"
"""

print("Metadata esperado:", VISUAL_METADATA_PATH)
print(EXAMPLE_METADATA_CSV)


## 10. Cargar y validar `metadata.csv` + imagenes

Si `metadata.csv` no existe todavia, se usa `metadata_template.csv` solo como referencia. Para entrenar, reemplazalo por `metadata.csv` con rutas reales.


In [ ]:
def load_visual_metadata(metadata_path: Path, trigger_word: str):
    path = metadata_path
    if not path.exists():
        template = metadata_path.parent / "metadata_template.csv"
        if template.exists():
            print(f"No existe {metadata_path}. Se usara template solo para referencia: {template}")
            path = template
        else:
            template.write_text(EXAMPLE_METADATA_CSV, encoding="utf-8")
            print(f"No existe {metadata_path}. Se creo template: {template}")
            return []

    df = pd.read_csv(path)
    required = {"file_path", "caption"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Faltan columnas en metadata visual: {sorted(missing)}")

    records = []
    errors = []
    root = path.parent
    for idx, row in df.iterrows():
        raw_path = Path(str(row["file_path"]))
        image_path = raw_path if raw_path.is_absolute() else (root / raw_path).resolve()
        caption = str(row["caption"]).strip()
        if image_path.suffix.lower() not in SUPPORTED_EXTENSIONS:
            errors.append((idx, f"extension no soportada: {image_path}"))
        if not image_path.exists():
            errors.append((idx, f"imagen no existe: {image_path}"))
        if not caption:
            errors.append((idx, "caption vacia"))
        if trigger_word not in caption:
            errors.append((idx, f"caption no contiene trigger word {trigger_word}"))
        records.append({"image": image_path, "caption": caption})

    if errors:
        print("Errores de metadata visual, mostrando maximo 10:")
        for err in errors[:10]:
            print(err)
        if path.name == "metadata_template.csv":
            print("Estos errores son esperados si aun no copiaste las imagenes reales.")
            return []
        raise ValueError(f"Metadata visual invalido: {len(errors)} errores.")

    print(f"Metadata visual usada: {path}")
    print(f"Imagenes validadas: {len(records)}")
    return records

visual_records = load_visual_metadata(VISUAL_METADATA_PATH, TRIGGER_WORD)
if visual_records:
    preview = []
    for record in visual_records[:5]:
        preview.append({"image": record["image"].name, "caption": record["caption"]})
        display(Image.open(record["image"]).resize((256, 256)))
    display(pd.DataFrame(preview))
else:
    print("Sin metadata visual real todavia. Copia imagenes y metadata.csv para entrenar DreamBooth LoRA.")


## 11. Preparar carpeta DreamBooth

El script oficial de DreamBooth LoRA SDXL usa una carpeta de imagenes y un `instance_prompt` comun. Las captions se guardan aparte para QA y trazabilidad.


In [ ]:
if visual_records:
    if TRAIN_IMAGES_DIR.exists():
        shutil.rmtree(TRAIN_IMAGES_DIR)
    TRAIN_IMAGES_DIR.mkdir(parents=True, exist_ok=True)

    dataset_summary = []
    for idx, record in enumerate(visual_records, start=1):
        src = record["image"]
        dst = TRAIN_IMAGES_DIR / f"{TRIGGER_WORD.lower()}_{idx:03d}.png"
        img = Image.open(src).convert("RGB")
        img.save(dst)
        dataset_summary.append({"file_name": dst.name, "source_path": str(src), "caption": record["caption"]})

    summary_df = pd.DataFrame(dataset_summary)
    summary_path = EVAL_OUTPUT_DIR / "visual_training_dataset_summary.csv"
    summary_df.to_csv(summary_path, index=False)
    print("Dataset DreamBooth preparado en:", TRAIN_IMAGES_DIR)
    print("Resumen:", summary_path)
    display(summary_df.head())
else:
    print("No hay imagenes reales validadas. Se omite preparacion DreamBooth.")


## 12. Descargar script oficial DreamBooth LoRA SDXL

El entrenamiento visual usa el script oficial de Diffusers.


In [ ]:
SCRIPT_PATH = OUTPUT_ROOT / "train_dreambooth_lora_sdxl.py"

if not SCRIPT_PATH.exists():
    !wget -q https://raw.githubusercontent.com/huggingface/diffusers/main/examples/dreambooth/train_dreambooth_lora_sdxl.py -O {SCRIPT_PATH}

print("Script DreamBooth:", SCRIPT_PATH)
print("Existe:", SCRIPT_PATH.exists())


## 13. Entrenar LoRA visual con Diffusers

Activa `RUN_VISUAL_TRAINING = True` cuando hayas cargado `metadata.csv` e imagenes reales.


In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

RESOLUTION = 512
MAX_TRAIN_STEPS = 250
VISUAL_LORA_RANK = 16
VISUAL_LEARNING_RATE = 1e-4
TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
MIXED_PRECISION = "fp16"

print({
    "model": SDXL_BASE_MODEL,
    "instance_prompt": INSTANCE_PROMPT,
    "resolution": RESOLUTION,
    "max_train_steps": MAX_TRAIN_STEPS,
    "rank": VISUAL_LORA_RANK,
    "learning_rate": VISUAL_LEARNING_RATE,
})


In [ ]:
if RUN_VISUAL_TRAINING:
    if not visual_records:
        raise RuntimeError("Carga metadata.csv e imagenes reales antes de entrenar LoRA visual.")
    if not torch.cuda.is_available():
        raise RuntimeError("Se requiere GPU para entrenar DreamBooth LoRA visual.")

    cmd = [
        "accelerate", "launch", str(SCRIPT_PATH),
        f"--pretrained_model_name_or_path={SDXL_BASE_MODEL}",
        f"--instance_data_dir={TRAIN_IMAGES_DIR}",
        f"--output_dir={VISUAL_LORA_OUTPUT_DIR}",
        f"--instance_prompt={INSTANCE_PROMPT}",
        f"--resolution={RESOLUTION}",
        f"--train_batch_size={TRAIN_BATCH_SIZE}",
        f"--gradient_accumulation_steps={GRADIENT_ACCUMULATION_STEPS}",
        f"--learning_rate={VISUAL_LEARNING_RATE}",
        "--lr_scheduler=constant",
        "--lr_warmup_steps=0",
        f"--max_train_steps={MAX_TRAIN_STEPS}",
        f"--rank={VISUAL_LORA_RANK}",
        f"--seed={SEED}",
        f"--mixed_precision={MIXED_PRECISION}",
        "--gradient_checkpointing",
    ]
    print("Ejecutando:")
    print(" ".join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True)
    print(result.stdout[-4000:])
    if result.returncode != 0:
        print(result.stderr[-4000:])
        raise RuntimeError("El entrenamiento LoRA visual fallo. Revisa GPU, versiones o rutas.")
else:
    print("RUN_VISUAL_TRAINING=False. Se omite entrenamiento visual.")

lora_files = sorted(VISUAL_LORA_OUTPUT_DIR.rglob("*.safetensors"))
print("LoRA visual files:", [str(p.relative_to(VISUAL_LORA_OUTPUT_DIR)) for p in lora_files])


## 14. Brief demo y prompt builder por placement

El brief alimenta al LLM fine-tuned. Si todavia no hay LLM adapter, el notebook usa un plan comercial fallback para poder construir prompts visuales.


In [ ]:
campaign_brief = {
    "brand": BRAND,
    "model_series": MODEL_SERIES,
    "trigger_word": TRIGGER_WORD,
    "campaign_goal": "Lead Generation",
    "vehicle_type": "hybrid SUV",
    "vehicle_model": "Nova Hybrid X",
    "price_range": "mid-range",
    "launch_message": "new model launch campaign",
    "target_audience": "Families 35-44",
    "customer_sector": "urban families",
    "location": "Miami",
    "language": "English",
    "duration": "30 Days",
    "preferred_channels": ["Instagram", "Facebook"],
    "promotion_type": "test drive + financing",
    "budget_hint": "$1,500",
    "market": "US and Latin America",
    "tone": "premium, confident, innovative",
}

fallback_commercial_plan = {
    "strategy": "Promote safety, family space, fuel efficiency and immediate test drive booking for urban families.",
    "channel_plan": "Use Instagram and Facebook for visual awareness, lead forms and remarketing.",
    "ad_copy": "Give your family more space, technology and efficiency. Book your test drive today.",
    "image_prompt": f"{TRIGGER_WORD} real car model in an English automotive dealership campaign for a mid-range hybrid SUV targeting urban families in Miami, premium commercial photography, bright city background, no readable text",
    "kpis": ["Leads", "Cost per Lead", "Test Drive Bookings", "Conversion Rate", "ROI"],
    "business_note": "Prioritize qualified leads and measure test drive bookings before scaling budget.",
}

DEFAULT_NEGATIVE_PROMPT = (
    "blurry, low quality, watermark, distorted text, malformed logo, extra wheels, "
    "deformed car, broken headlights, bad perspective, jpeg artifacts, people with distorted faces"
)

commercial_plan = fallback_commercial_plan
print(json.dumps(campaign_brief, indent=2))
print(json.dumps(commercial_plan, indent=2))


In [ ]:
PLACEMENTS = [
    {"name": "website_hero", "label": "website hero banner", "width": 768, "height": 432, "composition": "wide horizontal 16:9 layout with clean space for headline overlay"},
    {"name": "instagram_feed", "label": "Instagram feed ad", "width": 640, "height": 640, "composition": "square centered composition for social media feed"},
    {"name": "vertical_story", "label": "Instagram story or TikTok vertical ad", "width": 432, "height": 768, "composition": "vertical 9:16 mobile-first composition"},
    {"name": "showroom_poster", "label": "dealership showroom poster", "width": 640, "height": 832, "composition": "premium vertical print poster composition"},
    {"name": "highway_billboard", "label": "highway billboard", "width": 896, "height": 384, "composition": "ultra-wide billboard composition readable from distance"},
    {"name": "email_header", "label": "email campaign header", "width": 768, "height": 320, "composition": "clean wide email header composition"},
]

def build_visual_prompt(plan, brief, placement):
    base_prompt = plan.get("image_prompt") or fallback_commercial_plan["image_prompt"]
    return (
        f"{base_prompt}, {placement['label']}, {placement['composition']}, "
        f"{brief['tone']} tone, cinematic automotive photography, polished reflections, "
        "high quality commercial advertising, no readable text"
    )

prompts_df = pd.DataFrame([
    {
        "name": placement["name"],
        "label": placement["label"],
        "width": placement["width"],
        "height": placement["height"],
        "prompt": build_visual_prompt(commercial_plan, campaign_brief, placement),
        "negative_prompt": commercial_plan.get("negative_prompt") or DEFAULT_NEGATIVE_PROMPT,
        "seed": SEED + idx,
    }
    for idx, placement in enumerate(PLACEMENTS)
])

display(prompts_df[["name", "width", "height", "prompt", "negative_prompt", "seed"]])


## 15. Cargar SDXL base y LoRA visual

Si no hay LoRA visual entrenado todavia, esta seccion puede usar SDXL base para pruebas cuando `RUN_IMAGE_GENERATION=True`.


In [ ]:
if RUN_IMAGE_GENERATION:
    from diffusers import DPMSolverMultistepScheduler, StableDiffusionXLPipeline

    pipe = StableDiffusionXLPipeline.from_pretrained(
        SDXL_BASE_MODEL,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        use_safetensors=True,
        variant="fp16" if torch.cuda.is_available() else None,
    )
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    pipe = pipe.to("cuda" if torch.cuda.is_available() else "cpu")

    visual_lora_available = any(VISUAL_LORA_OUTPUT_DIR.rglob("*.safetensors"))
    if visual_lora_available:
        pipe.load_lora_weights(str(VISUAL_LORA_OUTPUT_DIR))
        print("SDXL + LoRA visual cargado desde", VISUAL_LORA_OUTPUT_DIR)
    else:
        print("No se encontro LoRA visual. Se generara con SDXL base.")
else:
    pipe = None
    print("RUN_IMAGE_GENERATION=False. Se omite carga de SDXL.")


## 16. Generar assets y metadata

Genera una pieza por placement y guarda PNG + metadata. Activa `RUN_IMAGE_GENERATION=True` cuando quieras producir imagenes.


In [ ]:
def generate_asset(row, use_lora=True):
    if pipe is None:
        raise RuntimeError("Carga primero SDXL con RUN_IMAGE_GENERATION=True.")

    has_lora = any(VISUAL_LORA_OUTPUT_DIR.rglob("*.safetensors"))
    if use_lora and has_lora:
        try:
            pipe.load_lora_weights(str(VISUAL_LORA_OUTPUT_DIR))
        except Exception:
            pass
        suffix = "lora"
    else:
        try:
            pipe.unload_lora_weights()
        except Exception:
            pass
        suffix = "base"

    device = "cuda" if torch.cuda.is_available() else "cpu"
    generator = torch.Generator(device=device).manual_seed(int(row["seed"]))
    start = time.perf_counter()
    image = pipe(
        prompt=row["prompt"],
        negative_prompt=row["negative_prompt"],
        width=int(row["width"]),
        height=int(row["height"]),
        num_inference_steps=30,
        guidance_scale=7.0,
        generator=generator,
    ).images[0]
    latency = time.perf_counter() - start
    image_path = GENERATED_ASSETS_DIR / f"{row['name']}_{suffix}_seed_{row['seed']}.png"
    image.save(image_path)
    return image, image_path, latency

if RUN_IMAGE_GENERATION:
    asset_records = []
    for _, row in prompts_df.iterrows():
        image, image_path, latency = generate_asset(row, use_lora=True)
        asset_records.append({
            "placement": row["name"],
            "path": str(image_path),
            "prompt": row["prompt"],
            "negative_prompt": row["negative_prompt"],
            "seed": int(row["seed"]),
            "width": int(row["width"]),
            "height": int(row["height"]),
            "model": SDXL_BASE_MODEL,
            "visual_lora": str(VISUAL_LORA_OUTPUT_DIR),
            "latency_sec": latency,
            "qualitative_note": "Review car identity consistency, commercial quality, placement fit and absence of distorted text.",
        })
        print(row["name"], image_path, f"{latency:.2f}s")
        display(image)

    metadata_path = EVAL_OUTPUT_DIR / "image_generation_metadata.json"
    metadata_path.write_text(json.dumps(asset_records, ensure_ascii=False, indent=2), encoding="utf-8")
    display(pd.DataFrame(asset_records))
    print("Metadata guardada en:", metadata_path)
else:
    print("RUN_IMAGE_GENERATION=False. Se omite generacion de assets.")


## 17. Comparacion SDXL base vs SDXL + LoRA visual

Usa mismo prompt y seed para evidenciar el efecto del LoRA visual.


In [ ]:
RUN_VISUAL_COMPARISON = False

if RUN_VISUAL_COMPARISON:
    if pipe is None:
        raise RuntimeError("Carga SDXL primero con RUN_IMAGE_GENERATION=True.")
    if not any(VISUAL_LORA_OUTPUT_DIR.rglob("*.safetensors")):
        raise RuntimeError("No hay LoRA visual entrenado para comparar.")

    comparison_rows = prompts_df.head(3).copy()
    comparison_records = []

    for _, row in comparison_rows.iterrows():
        base_image, base_path, base_latency = generate_asset(row, use_lora=False)
        lora_image, lora_path, lora_latency = generate_asset(row, use_lora=True)
        comparison_records.append({
            "placement": row["name"],
            "base_path": str(base_path),
            "lora_path": str(lora_path),
            "prompt": row["prompt"],
            "seed": int(row["seed"]),
            "base_latency_sec": base_latency,
            "lora_latency_sec": lora_latency,
        })

        canvas = Image.new("RGB", (base_image.width + lora_image.width, max(base_image.height, lora_image.height)), "white")
        canvas.paste(base_image, (0, 0))
        canvas.paste(lora_image, (base_image.width, 0))
        display(canvas)

    comparison_path = EVAL_OUTPUT_DIR / "sdxl_base_vs_visual_lora_comparison.json"
    comparison_path.write_text(json.dumps(comparison_records, ensure_ascii=False, indent=2), encoding="utf-8")
    display(pd.DataFrame(comparison_records))
    print("Comparacion guardada en:", comparison_path)
else:
    print("RUN_VISUAL_COMPARISON=False. Se omite comparacion visual.")


## 18. Reporte final y ROI

El entregable final debe incluir:

- JSON SFT validado con 200+ ejemplos.
- Adapter LLM en `outputs/commercial-qwen-lora/`.
- Comparacion LLM base vs fine-tuned.
- `metadata.csv` visual validado.
- Adapter visual en `outputs/automotive-lora/`.
- Comparacion SDXL base vs SDXL + LoRA visual.
- Assets PNG por placement y metadata.

Formula de ROI:

```text
ROI estimado = ((horas creativas ahorradas * costo hora equipo creativo * campanas mensuales) + (horas comerciales ahorradas * costo hora comercial * propuestas mensuales) - costo operativo IA) / costo operativo IA
```

Ejemplo:

```text
ahorro creativo mensual = 12 * 6 * 35 = USD 2,520
ahorro comercial mensual = 2 * 40 * 25 = USD 2,000
costo IA mensual = USD 300
ROI = (2,520 + 2,000 - 300) / 300 = 14.07x
```

Limitaciones: calidad dependiente de datos, posible deformacion de texto/logos, necesidad de revision humana antes de uso comercial y posible ejecucion separada de entrenamientos por limites de GPU.
